In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
crop_data = pd.read_csv("crop_production.csv")
crop_data = crop_data.dropna()

# Add Yield column
crop_data["Yield"] = crop_data["Production"] / crop_data["Area"]

print(f"Dataset shape: {crop_data.shape}")
print(crop_data.head())

Dataset shape: (242361, 8)
                    State_Name District_Name  Crop_Year       Season  \
0  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
1  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
2  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
3  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    
4  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    

                  Crop    Area  Production     Yield  
0             Arecanut  1254.0      2000.0  1.594896  
1  Other Kharif pulses     2.0         1.0  0.500000  
2                 Rice   102.0       321.0  3.147059  
3               Banana   176.0       641.0  3.642045  
4            Cashewnut   720.0       165.0  0.229167  


In [ ]:
le = LabelEncoder()
crop_data["State_Enc"]  = le.fit_transform(crop_data["State_Name"])
crop_data["Season_Enc"] = le.fit_transform(crop_data["Season"].str.strip())
crop_data["Crop_Enc"]   = le.fit_transform(crop_data["Crop"])

In [ ]:
FEATURES = ["State_Enc", "Season_Enc", "Crop_Enc", "Area"]

X            = crop_data[FEATURES]
y_yield      = crop_data["Yield"]
y_production = crop_data["Production"]

In [ ]:
X_train, X_test, yY_train, yY_test = train_test_split(X, y_yield,      test_size=0.2, random_state=42)
_,       _,      yP_train, yP_test = train_test_split(X, y_production,  test_size=0.2, random_state=42)

In [ ]:
params = dict(
    n_estimators  = 300,
    learning_rate = 0.1,
    max_depth     = 5,
    random_state  = 42,
    n_jobs        = -1,
)

In [ ]:
model_yield = xgb.XGBRegressor(**params)
model_yield.fit(X_train, yY_train)

# Production Model
model_prod = xgb.XGBRegressor(**params)
model_prod.fit(X_train, yP_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
def evaluate(name, y_true, y_pred):
    print(f"\n{name} Results:")
    print(f"  MAE  : {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"  RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"  R²   : {r2_score(y_true, y_pred):.4f}")

evaluate("Yield",      yY_test, model_yield.predict(X_test))
evaluate("Production", yP_test, model_prod.predict(X_test))


Yield Results:
  MAE  : 21.7389
  RMSE : 285.0369
  R²   : 0.8498

Production Results:
  MAE  : 206854.8931
  RMSE : 4588773.8511
  R²   : 0.9477


In [ ]:
print("\nSample Predictions vs Actuals:")
sample = pd.DataFrame({
    "Actual_Yield":         yY_test.values[:10],
    "Predicted_Yield":      model_yield.predict(X_test)[:10],
    "Actual_Production":    yP_test.values[:10],
    "Predicted_Production": model_prod.predict(X_test)[:10],
}).round(2)
print(sample.to_string(index=False))


Sample Predictions vs Actuals:
 Actual_Yield  Predicted_Yield  Actual_Production  Predicted_Production
         0.83             0.32             4529.0          21922.929688
         0.51             0.17              425.0          13753.219727
        28.22            49.91              254.0         -21803.699219
         0.54            46.91                7.0         -49329.421875
         1.11             0.64              130.0            865.919983
         0.40             2.07               21.0            527.369995
         1.58             4.13            98392.0         165457.828125
         0.25             2.65              182.0          -3542.979980
         9.60            -0.41              835.0         -98511.429688
         0.52            -0.21              190.0          -8212.889648


In [ ]:
print("States  :", crop_data["State_Name"].unique())
print("Crops   :", crop_data["Crop"].unique())
print("Seasons :", crop_data["Season"].str.strip().unique())

States  : ['Andaman and Nicobar Islands' 'Andhra Pradesh' 'Arunachal Pradesh'
 'Assam' 'Bihar' 'Chandigarh' 'Chhattisgarh' 'Dadra and Nagar Haveli'
 'Goa' 'Gujarat' 'Haryana' 'Himachal Pradesh' 'Jammu and Kashmir '
 'Jharkhand' 'Karnataka' 'Kerala' 'Madhya Pradesh' 'Maharashtra' 'Manipur'
 'Meghalaya' 'Mizoram' 'Nagaland' 'Odisha' 'Puducherry' 'Punjab'
 'Rajasthan' 'Sikkim' 'Tamil Nadu' 'Telangana ' 'Tripura' 'Uttar Pradesh'
 'Uttarakhand' 'West Bengal']
Crops   : ['Arecanut' 'Other Kharif pulses' 'Rice' 'Banana' 'Cashewnut' 'Coconut '
 'Dry ginger' 'Sugarcane' 'Sweet potato' 'Tapioca' 'Black pepper'
 'Dry chillies' 'other oilseeds' 'Turmeric' 'Maize' 'Moong(Green Gram)'
 'Urad' 'Arhar/Tur' 'Groundnut' 'Sunflower' 'Bajra' 'Castor seed'
 'Cotton(lint)' 'Horse-gram' 'Jowar' 'Korra' 'Ragi' 'Tobacco' 'Gram'
 'Wheat' 'Masoor' 'Sesamum' 'Linseed' 'Safflower' 'Onion'
 'other misc. pulses' 'Samai' 'Small millets' 'Coriander' 'Potato'
 'Other  Rabi pulses' 'Soyabean' 'Beans & Mutter(Vegetable)'

In [ ]:
# ── 9. Simple User Prediction ───────────────────────────

print("\n--- Crop Prediction ---")

# Take inputs
state  = input("State: ").strip()
crop   = input("Crop: ").strip()
season = input("Season: ").strip()
year   = int(input("Year: "))
area   = float(input("Area (hectares): "))

try:
    # Encode inputs
    state_enc  = le_state.transform([state])[0]
    crop_enc   = le_crop.transform([crop])[0]
    season_enc = le_season.transform([season])[0]

    # Create input
    user_data = pd.DataFrame([[state_enc, season_enc, crop_enc, year, area]],
                            columns=FEATURES)

    # Predict
    yield_pred = model_yield.predict(user_data)[0]
    prod_pred  = model_prod.predict(user_data)[0]
    # Output
    print("\n--- Result ---")
    print(f"Yield      : {yield_pred:.2f}")
    print(f"Production : {prod_pred:.2f}")

except:
    print("\nInvalid input! Please enter values exactly as in dataset.")


--- Crop Prediction ---
